# Stats Pipeline — Jersey Teams, Passes, Goals (Kaggle)

Runs YOLO detection + tracking + stats on a football clip.

## Setup
1. **GPU** optional (CPU works, slower)
2. **Add Input** → **Models** → your trained `best.pt`
3. **Add Input** → **Dataset** → upload your test `.mp4` (or use a Kaggle dataset)
4. Run all cells

## What it detects
- **Teams** — KMeans jersey color clustering (saturated vs white kits)
- **Passes** — low ball speed (player A) → high speed (in flight) → low speed (player B)
- **Goals** — ball crosses goal mouth line from detected goalposts

In [ ]:
!pip install -q ultralytics supervision scikit-learn opencv-python-headless tqdm

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv
from sklearn.cluster import KMeans

# --- paths (edit VIDEO_PATH if needed) ---
MODEL_PATH = next(Path("/kaggle/input").rglob("best.pt"), Path("best.pt"))
VIDEO_PATH = next(Path("/kaggle/input").rglob("*.mp4"), None)
if VIDEO_PATH is None:
    raise FileNotFoundError("Add a .mp4 video as Kaggle dataset input")
OUT_DIR = Path("/kaggle/working/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Model:", MODEL_PATH)
print("Video:", VIDEO_PATH)

In [ ]:
# --- Jersey KMeans (inline, same logic as analysis/player_color_assignment.py) ---

def extract_jersey_hsv(frame, bbox):
    x1, y1, x2, y2 = map(int, bbox)
    h = max(1, y2 - y1)
    crop = frame[y1:y1 + h // 2, x1:x2]
    if crop.size == 0:
        return None
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    return np.median(hsv.reshape(-1, 3), axis=0).astype(np.float32)

def fit_team_ids(track_colors: dict, barcelona_team_id=0):
  """track_colors: {track_id: [hsv arrays]}"""
    means = {tid: np.mean(v, axis=0) for tid, v in track_colors.items() if len(v) >= 3}
    if len(means) < 2:
        return {}
    ids = list(means.keys())
    X = np.array([means[i] for i in ids])
    labels = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X)
    sats = [np.median(X[labels == c][:, 1]) for c in (0, 1)]
    barca_cluster = 0 if sats[0] >= sats[1] else 1
    real_id = 1 - barcelona_team_id
    out = {}
    for tid, lab in zip(ids, labels):
        out[tid] = barcelona_team_id if lab == barca_cluster else real_id
    return out

In [ ]:
# --- Pass + goal detectors (inline) ---

def foot(bbox):
    return ((bbox[0] + bbox[2]) / 2, bbox[3])

class PassDetector:
    def __init__(self):
        self.events = []
        self.state = "idle"
        self.passer_tid = self.passer_team = None
        self.passer_pos = None
        self.start = 0
        self.peak = 0
        self.last_pass = -99

    def closest(self, players, ball, R=90):
        best, bd = None, 1e9
        for p in players:
            d = np.hypot(foot(p["bbox"])[0] - ball[0], foot(p["bbox"])[1] - ball[1])
            if d < bd:
                bd, best = d, p
        return best if best and bd <= R else None

    def update(self, fi, ball, speed, players):
        if ball is None:
            return
        owner = self.closest(players, ball)
        if self.state in ("idle", "controlled"):
            if owner and speed <= 8:
                self.state = "controlled"
                self.passer_tid = owner["track_id"]
                self.passer_team = owner["team_id"]
                self.passer_pos = foot(owner["bbox"])
                self.start = fi
            elif self.state == "controlled" and speed >= 18:
                self.state = "in_flight"
                self.peak = speed
            return
        if self.state == "in_flight":
            self.peak = max(self.peak, speed)
            if fi - self.start > 90:
                self.state = "idle"
                return
            if owner and speed <= 8 and owner["track_id"] != self.passer_tid and owner["team_id"] == self.passer_team and fi - self.last_pass >= 8:
                if np.hypot(owner["bbox"][0] - self.passer_pos[0], owner["bbox"][1] - self.passer_pos[1]) >= 40:
                    self.events.append({"frame": fi, "team": self.passer_team, "from": self.passer_tid, "to": owner["track_id"]})
                    self.last_pass = fi
                    self.state = "controlled"
                    self.passer_tid = owner["track_id"]
                    self.passer_pos = foot(owner["bbox"])

class GoalDetector:
    def __init__(self, goal_boxes, goal_lines):
        self.goal_boxes = goal_boxes
        self.goal_lines = goal_lines
        self.events = []
        self.prev_ball = None
        self.last_goal = -99

    def crossed(self, prev, curr, side):
        line = self.goal_lines[side]
        mx, y1, y2 = line["mouth_x"], line["y1"], line["y2"]
        if not (min(y1, y2) <= curr[1] <= max(y1, y2)):
            return False
        return (prev[0] > mx >= curr[0]) if side == "team0" else (prev[0] < mx <= curr[0])

    def update(self, fi, ball, speed):
        if ball is None:
            self.prev_ball = None
            return
        if self.prev_ball is None:
            self.prev_ball = ball
            return
        prev = self.prev_ball
        self.prev_ball = ball
        if speed < 12 or fi - self.last_goal < 60:
            return
        for side in self.goal_boxes:
            if self.crossed(prev, ball, side):
                scoring = 1 if side == "team0" else 0
                self.events.append({"frame": fi, "scoring_team": scoring, "goal": side})
                self.last_goal = fi
                return

In [ ]:
# --- Run YOLO + ByteTrack on video ---

model = YOLO(str(MODEL_PATH))
tracker = sv.ByteTrack()
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

track_colors = {}
goalpost_boxes = {"team0": [], "team1": []}
frames_data = []
fi = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break
    r = model.predict(frame, conf=0.1, verbose=False)[0]
    dets = sv.Detections.from_ultralytics(r)
    tracked = tracker.update_with_detections(dets)

    ball = None
    players = []
    goalposts = []
    names = r.names

    for i in range(len(tracked)):
        cid = int(tracked.class_id[i])
        cname = names[cid]
        bbox = tracked.xyxy[i].tolist()
        tid = int(tracked.tracker_id[i]) if tracked.tracker_id[i] is not None else None
        if cname == "ball":
            ball = ((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)
        elif cname in ("player", "goalkeeper") and tid is not None:
            players.append({"track_id": tid, "bbox": bbox, "team_id": None})
            col = extract_jersey_hsv(frame, bbox)
            if col is not None:
                track_colors.setdefault(tid, []).append(col)
        elif cname == "goalpost":
            cx = (bbox[0] + bbox[2]) / 2
            goalposts.append({"bbox": bbox, "cx": cx})

    if goalposts:
        mid = w / 2
        for g in goalposts:
            key = "team0" if g["cx"] < mid else "team1"
            goalpost_boxes[key].append(g["bbox"])

    frames_data.append({"frame": fi, "ball": ball, "players": players})
    fi += 1
cap.release()
print(f"Processed {fi} frames")

In [ ]:
# --- Build goal regions from goalposts ---

def merge_boxes(boxes, pad=40):
    if not boxes:
        return None
    x1 = min(b[0] for b in boxes) - pad
    y1 = min(b[1] for b in boxes) - pad
    x2 = max(b[2] for b in boxes) + pad
    y2 = max(b[3] for b in boxes) + pad
    return [x1, y1, x2, y2]

goal_boxes = {
    "team0": merge_boxes(goalpost_boxes["team0"]) or [0, h * 0.35, w * 0.08, h * 0.65],
    "team1": merge_boxes(goalpost_boxes["team1"]) or [w * 0.92, h * 0.35, w, h * 0.65],
}
goal_lines = {
    "team0": {"mouth_x": goal_boxes["team0"][2], "y1": goal_boxes["team0"][1], "y2": goal_boxes["team0"][3]},
    "team1": {"mouth_x": goal_boxes["team1"][0], "y1": goal_boxes["team1"][1], "y2": goal_boxes["team1"][3]},
}
print("Goal boxes:", goal_boxes)

In [ ]:
# --- Assign teams + run pass/goal detectors ---

team_map = fit_team_ids(track_colors, barcelona_team_id=0)
print("Team map:", team_map)

pass_det = PassDetector()
goal_det = GoalDetector(goal_boxes, goal_lines)
possession = {"team0": 0, "team1": 0, "loose": 0}
prev_ball = None
last_touch = None

for rec in frames_data:
    fi = rec["frame"]
    ball = rec["ball"]
    players = []
    for p in rec["players"]:
        tid = p["track_id"]
        if tid in team_map:
            p = dict(p)
            p["team_id"] = team_map[tid]
            players.append(p)

    speed = 0.0
    if ball and prev_ball:
        speed = np.hypot(ball[0] - prev_ball[0], ball[1] - prev_ball[1])
    if ball:
        prev_ball = ball

    owner = pass_det.closest(players, ball) if ball else None
    if owner:
        possession[f"team{owner['team_id']}"] += 1
        last_touch = owner["team_id"]
    elif ball and last_touch is not None:
        possession[f"team{last_touch}"] += 1
    elif ball:
        possession["loose"] += 1

    pass_det.update(fi, ball, speed, players)
    goal_det.update(fi, ball, speed)

total = sum(possession.values()) or 1
stats = {
    "team_names": {"team0": "barcelona", "team1": "real_madrid"},
    "possession": {k: v / total * 100 for k, v in possession.items()},
    "passes": {"team0": sum(1 for e in pass_det.events if e["team"] == 0),
               "team1": sum(1 for e in pass_det.events if e["team"] == 1)},
    "pass_events": pass_det.events,
    "goals": {"team0": sum(1 for e in goal_det.events if e["scoring_team"] == 0),
              "team1": sum(1 for e in goal_det.events if e["scoring_team"] == 1)},
    "goal_events": goal_det.events,
    "goal_boxes": goal_boxes,
    "goal_lines": goal_lines,
}

out = OUT_DIR / "stats.json"
out.write_text(json.dumps(stats, indent=2))
print(json.dumps({k: stats[k] for k in ["possession", "passes", "goals", "pass_events", "goal_events"]}, indent=2))
print("Saved", out)